In [ ]:
# Cell 1 - Setup. No extra installs needed - just torch, already available.
from google.colab import drive
drive.mount('/content/drive')
import os
os.chdir('/content/drive/MyDrive/ML_final')

import torch
print('torch:', torch.__version__, '| CUDA:', torch.cuda.is_available())

import pandas as pd
import numpy as np

In [ ]:
!pip install -q dagshub mlflow

import dagshub
import mlflow

from mlflow import MlflowClient
from mlflow.entities import ViewType


# Connect to the DagsHub MLflow server
dagshub.init(
    repo_owner="skupr23",
    repo_name="ml_final",
    mlflow=True,
)

EXPERIMENT_NAME = "DLinear_Training"


# Close a run left open by an interrupted notebook execution
if mlflow.active_run() is not None:
    mlflow.end_run()


client = MlflowClient()

# Include active and deleted experiments in the search
matches = client.search_experiments(
    view_type=ViewType.ALL,
    filter_string=f"name = '{EXPERIMENT_NAME}'",
)

if matches:
    experiment = matches[0]

    if experiment.lifecycle_stage == "deleted":
        client.restore_experiment(experiment.experiment_id)
        print(
            f"Restored deleted experiment: "
            f"{EXPERIMENT_NAME} ({experiment.experiment_id})"
        )

# Creates the experiment only if it genuinely does not exist
mlflow.set_experiment(EXPERIMENT_NAME)

active_experiment = client.get_experiment_by_name(EXPERIMENT_NAME)

print("Tracking URI :", mlflow.get_tracking_uri())
print("Experiment   :", active_experiment.name)
print("Experiment ID:", active_experiment.experiment_id)
print("Lifecycle    :", active_experiment.lifecycle_stage)

# Cell 2 - Load processed data (identical source as the other notebooks)

In [ ]:
train = pd.read_pickle('data/train_processed.pkl')
train = train.sort_values(['Store','Dept','Date'])
print(train['Date'].min(), train['Date'].max())

# Cell 3 - WMAE metric (same scoring function used everywhere)

In [ ]:
def wmae(y_true, y_pred, is_holiday):
    weights = np.where(is_holiday, 5, 1)
    return np.sum(weights * np.abs(y_true - y_pred)) / np.sum(weights)

# Cell 4 - Rebuild the XGBoost / LightGBM holdout

DLinear is scored on the **same rows** as the GBM notebooks: every Store-Dept
observation in the window `Date >= max(Date) - 39 weeks`, which is 40 weekly
dates from 2012-01-27 to 2012-10-26. No minimum-length filter on the scored rows,
and no interpolated values enter the score - a forecast is only ever compared
against a real observation.

**The baseline is the population check.** It depends only on which rows are being
scored, not on DLinear, so it must reproduce the `1789.9133525504185` printed by
both GBM notebooks. If it does, `final_wmae` below is directly comparable to
XGBoost's 1281.08 and LightGBM's 1297.44.

In [ ]:
VAL_WEEKS = 39          # the GBM notebooks' window; yields 40 weekly dates

_t = train.sort_values('Date')
VAL_START = _t['Date'].max() - pd.Timedelta(weeks=VAL_WEEKS)
gbm_val = _t[_t['Date'] >= VAL_START].copy()
gbm_fit = _t[_t['Date'] < VAL_START].copy()

FIT_END = gbm_fit['Date'].max()
val_dates = pd.DatetimeIndex(np.sort(gbm_val['Date'].unique()))

print('fit ends:  ', FIT_END)
print('val range: ', gbm_val['Date'].min(), 'to', gbm_val['Date'].max())
print('val rows:  ', len(gbm_val),
      '| val dates:', len(val_dates),
      '| val pairs:', gbm_val[['Store','Dept']].drop_duplicates().shape[0])

# --- population check: this MUST reproduce the GBM notebooks' baseline ---
baseline_pred = gbm_val['lag_52'].fillna(gbm_val['storedept_median_sales'])
baseline_wmae = wmae(gbm_val['Weekly_Sales'], baseline_pred, gbm_val['IsHoliday'])
print('\n52-week seasonal baseline WMAE:', baseline_wmae)
print('   XGBoost / LightGBM reported:  1789.9133525504185')
if abs(baseline_wmae - 1789.9133525504185) > 1e-6:
    raise AssertionError('population mismatch: this is not the GBM validation set')

# --- leak-free fallbacks for pairs with too little context, from the fit portion only ---
sd_median = gbm_fit.groupby(['Store','Dept'])['Weekly_Sales'].median().to_dict()
d_median = gbm_fit.groupby('Dept')['Weekly_Sales'].median().to_dict()
g_median = float(gbm_fit['Weekly_Sales'].median())

# Cell 5 - Per-series context arrays, built from the fit portion only

DLinear consumes a fixed-length window of consecutive weeks, so each series has to
sit on a regular `W-FRI` grid. Gaps are interpolated - but note the asymmetry:
**interpolated values are used only as model input, never as a scoring target.**
Every series is reindexed to end at the same cutoff (`FIT_END`), so the 52-week
context all series present to the model ends on the same date.

A series needs at least `INPUT_LEN` real observations in the fit portion to get a
context window. Pairs that do not clear that bar fall back to a median in Cell 9.

In [ ]:
INPUT_LEN = 52
OUTPUT_LEN = 13

series_ctx = {}
n_ctx_interpolated = 0

for (store, dept), g in gbm_fit.groupby(['Store','Dept'], sort=False):
    if len(g) < INPUT_LEN:
        continue
    s = g.sort_values('Date').set_index('Date')['Weekly_Sales']
    # one regular weekly grid per series, every series ending at FIT_END
    full_idx = pd.date_range(s.index.min(), FIT_END, freq='W-FRI')
    s = s.reindex(full_idx)
    n_ctx_interpolated += int(s.isna().sum())
    s = s.interpolate().bfill().ffill()
    series_ctx[(store, dept)] = s.to_numpy(dtype=np.float32)

print('Series with a usable context window:', len(series_ctx))
print('Interpolated context points (model input only, never scored):', n_ctx_interpolated)

# Cell 6 - Sliding (input, output) windows from the fit portion

The last window of each series is held out as a monitor set for early stopping,
the same role the monitor set plays in the PatchTST notebook. Nothing here can
see the validation window: `series_ctx` was built from `gbm_fit` alone.

In [ ]:
X_train, Y_train, X_monitor, Y_monitor = [], [], [], []

for key, arr in series_ctx.items():
    n = len(arr)
    max_start = n - INPUT_LEN - OUTPUT_LEN
    if max_start < 1:
        continue
    for start in range(max_start):          # 0 .. max_start-1 -> training
        X_train.append(arr[start:start+INPUT_LEN])
        Y_train.append(arr[start+INPUT_LEN:start+INPUT_LEN+OUTPUT_LEN])
    X_monitor.append(arr[max_start:max_start+INPUT_LEN])
    Y_monitor.append(arr[max_start+INPUT_LEN:max_start+INPUT_LEN+OUTPUT_LEN])

X_train = np.stack(X_train).astype(np.float32)
Y_train = np.stack(Y_train).astype(np.float32)
X_monitor = np.stack(X_monitor).astype(np.float32)
Y_monitor = np.stack(Y_monitor).astype(np.float32)

print('Train windows:', X_train.shape, '| Monitor windows:', X_monitor.shape)

# Cell 7 - DLinear implementation in plain PyTorch

DLinear (Zeng et al., *Are Transformers Effective for Time Series Forecasting?*)
is deliberately tiny: decompose the lookback window into a trend and a seasonal
remainder, run **one linear layer over each**, and add them. No attention, no
recurrence, no hidden nonlinearity.

* `MovingAvg` - a stride-1 average pool with the endpoints replicated on both
  sides, so the trend comes out the same length as the input.
* `SeriesDecomp` - `trend = movavg(x)`, `seasonal = x - trend`.
* Two `Linear(input_len, output_len)` heads, summed.

One deviation from the paper: the instance normalization (RevIN) below is not part
of DLinear as published, which standardizes globally instead. It is kept here so
that DLinear and PatchTST differ **only in the encoder**, which is exactly the
comparison the paper is arguing about. Both models therefore see identically
normalized inputs.

In [ ]:
import torch.nn as nn

KERNEL_SIZE = 25   # moving-average window for the trend component (must be odd)


class RevIN(nn.Module):
    """Instance normalization: subtract/divide by each window's own
    mean/std before the model, then undo it on the output."""
    def __init__(self, eps=1e-5):
        super().__init__()
        self.eps = eps

    def forward(self, x):
        # x: (batch, input_len)
        mean = x.mean(dim=1, keepdim=True)
        std = x.std(dim=1, keepdim=True) + self.eps
        x_norm = (x - mean) / std
        return x_norm, mean, std


class MovingAvg(nn.Module):
    """Stride-1 moving average with replicated endpoints, length-preserving."""
    def __init__(self, kernel_size):
        super().__init__()
        assert kernel_size % 2 == 1, 'kernel_size must be odd to preserve length'
        self.kernel_size = kernel_size
        self.pad = (kernel_size - 1) // 2
        self.avg = nn.AvgPool1d(kernel_size=kernel_size, stride=1, padding=0)

    def forward(self, x):
        # x: (batch, input_len) -> (batch, input_len)
        front = x[:, :1].repeat(1, self.pad)
        end = x[:, -1:].repeat(1, self.pad)
        padded = torch.cat([front, x, end], dim=1).unsqueeze(1)
        return self.avg(padded).squeeze(1)


class SeriesDecomp(nn.Module):
    """Split a window into a moving-average trend and the seasonal remainder."""
    def __init__(self, kernel_size):
        super().__init__()
        self.moving_avg = MovingAvg(kernel_size)

    def forward(self, x):
        trend = self.moving_avg(x)
        seasonal = x - trend
        return seasonal, trend


class DLinear(nn.Module):
    def __init__(self, input_len, output_len, kernel_size=KERNEL_SIZE):
        super().__init__()
        self.revin = RevIN()
        self.decomp = SeriesDecomp(kernel_size)
        self.linear_seasonal = nn.Linear(input_len, output_len)
        self.linear_trend = nn.Linear(input_len, output_len)

    def forward(self, x):
        # x: (batch, input_len)
        x_norm, mean, std = self.revin(x)
        seasonal, trend = self.decomp(x_norm)
        out_norm = self.linear_seasonal(seasonal) + self.linear_trend(trend)
        out = out_norm * std + mean       # undo RevIN
        return out


device = 'cuda' if torch.cuda.is_available() else 'cpu'
model_dlinear = DLinear(INPUT_LEN, OUTPUT_LEN).to(device)
print(model_dlinear)
print('Trainable parameters:', sum(p.numel() for p in model_dlinear.parameters()))

# Cell 8 - Train with early stopping on the monitor set

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

X_train_t = torch.from_numpy(X_train)
Y_train_t = torch.from_numpy(Y_train)
X_monitor_t = torch.from_numpy(X_monitor).to(device)
Y_monitor_t = torch.from_numpy(Y_monitor).to(device)

train_loader = DataLoader(TensorDataset(X_train_t, Y_train_t), batch_size=1024, shuffle=True)

loss_fn = torch.nn.HuberLoss(delta=1.0)
optimizer = torch.optim.AdamW(model_dlinear.parameters(), lr=5e-4, weight_decay=1e-2)
N_EPOCHS = 30
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=N_EPOCHS)

best_val_loss = float('inf')
patience, patience_counter = 5, 0
best_state = None
best_epoch = 0
history = []   # per-epoch losses, replayed into MLflow in the logging cell

for epoch in range(N_EPOCHS):
    model_dlinear.train()
    train_loss_sum, n_batches = 0.0, 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        pred = model_dlinear(xb)
        loss = loss_fn(pred, yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_dlinear.parameters(), 0.5)
        optimizer.step()
        train_loss_sum += loss.item()
        n_batches += 1
    scheduler.step()

    model_dlinear.eval()
    with torch.no_grad():
        val_pred = model_dlinear(X_monitor_t)
        val_loss = loss_fn(val_pred, Y_monitor_t).item()

    train_loss = train_loss_sum / n_batches
    history.append({
        'epoch': epoch + 1,
        'train_loss': train_loss,
        'val_loss': val_loss,
        'lr': scheduler.get_last_lr()[0],
    })

    print(f"Epoch {epoch+1} - train_loss: {train_loss:.5f} - val_loss: {val_loss:.5f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = {k: v.clone() for k, v in model_dlinear.state_dict().items()}
        best_epoch = epoch + 1
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print("Early stopping.")
            break

epochs_trained = epoch + 1
model_dlinear.load_state_dict(best_state)

# Cell 9 - Forecast the 40 validation dates

The head emits 13 weeks at a time, so the block is rolled forward
`ceil(40 / 13) = 4` times, feeding predictions back in as context, then truncated
to the 40 dates. Predictions are indexed **by date** and reindexed onto each
pair's real validation rows, so a pair with gapped weeks is scored only on the
weeks it actually has.

Pairs without a context window fall back to the fit-portion median, the same
store-dept -> dept -> global ladder the Prophet notebook uses.

In [ ]:
N_BLOCKS = int(np.ceil(len(val_dates) / OUTPUT_LEN))
print('Rolling the', OUTPUT_LEN, 'week head forward', N_BLOCKS, 'times ->',
      N_BLOCKS * OUTPUT_LEN, 'weeks, truncated to', len(val_dates))

model_dlinear.eval()
gbm_val = gbm_val.sort_values(['Store','Dept','Date'])

pred_parts = []
n_model = n_fallback = 0

with torch.no_grad():
    for (store, dept), vg in gbm_val.groupby(['Store','Dept'], sort=False):
        arr = series_ctx.get((store, dept))

        if arr is None or len(arr) < INPUT_LEN:
            v = sd_median.get((store, dept), np.nan)
            if not np.isfinite(v):
                v = d_median.get(dept, np.nan)
            if not np.isfinite(v):
                v = g_median
            by_date = pd.Series(np.full(len(val_dates), float(v)), index=val_dates)
            n_fallback += 1
        else:
            context = arr[-INPUT_LEN:].copy()
            preds = []
            for _ in range(N_BLOCKS):
                x = torch.from_numpy(context[-INPUT_LEN:]).unsqueeze(0).to(device)
                block = model_dlinear(x).cpu().numpy().flatten()
                preds.extend(block)
                context = np.concatenate([context, block])
            by_date = pd.Series(np.asarray(preds[:len(val_dates)]), index=val_dates)
            n_model += 1

        # reindex by date: a pair is scored only on the weeks it really has
        pred_parts.append(pd.Series(by_date.reindex(vg['Date']).to_numpy(), index=vg.index))

all_pred_s = pd.concat(pred_parts)
scored = gbm_val.loc[all_pred_s.index]
all_pred = all_pred_s.to_numpy()
all_true = scored['Weekly_Sales'].to_numpy()
all_holiday = scored['IsHoliday'].to_numpy().astype(bool)

population_complete = len(scored) == len(gbm_val)
assert np.isfinite(all_pred).all(), 'a validation row did not receive a prediction'

print(f'\nDLinear forecasts: {n_model} pairs | median fallbacks: {n_fallback} pairs')
print('Rows scored:', len(scored), '| full population:', population_complete)

# Cell 10 - Evaluate

In [ ]:
dlinear_wmae = wmae(all_true, all_pred, all_holiday)
dlinear_wmae_holiday = wmae(all_true[all_holiday], all_pred[all_holiday], all_holiday[all_holiday])
dlinear_wmae_non_holiday = wmae(all_true[~all_holiday], all_pred[~all_holiday], all_holiday[~all_holiday])

# per-pair breakdown: WMAE per pair = sum(w * |err|) / sum(w), computed vectorized
_s = scored.assign(pred=all_pred)
_s['_w'] = np.where(all_holiday, 5, 1)
_s['_wae'] = _s['_w'] * (_s['Weekly_Sales'] - _s['pred']).abs()
per_pair = (
    _s.groupby(['Store','Dept'])
      .agg(_wae_sum=('_wae','sum'),
           _w_sum=('_w','sum'),
           mean_weekly_sales=('Weekly_Sales','mean'),
           n_val_weeks=('Weekly_Sales','size'))
      .reset_index()
)
per_pair['wmae'] = per_pair['_wae_sum'] / per_pair['_w_sum']
per_pair = (per_pair[['Store','Dept','wmae','mean_weekly_sales','n_val_weeks']]
            .sort_values('wmae')
            .reset_index(drop=True))

print('Baseline (52wk):  ', baseline_wmae)
print('DLinear WMAE:     ', dlinear_wmae)
print('  XGBoost:        ', 1281.080964551936)
print('  LightGBM:       ', 1297.4414789901796)
print('\nHoliday WMAE:    ', dlinear_wmae_holiday)
print('Non-holiday WMAE:', dlinear_wmae_non_holiday)

# Cell 11 - Save model locally (same pattern as the other notebooks)

In [ ]:
import os
import joblib

os.makedirs('models', exist_ok=True)
joblib.dump(model_dlinear.cpu(), 'models/dlinear_best.pkl')
print('Saved.')

In [ ]:
# ============================================================
# MLflow - one experiment per architecture, one run per stage
# ============================================================
import os
import tempfile

import cloudpickle
import mlflow.pytorch

PREPROCESSING_STEPS = [
    {'step': 'load_processed', 'detail': 'read data/train_processed.pkl, sort by Store-Dept-Date'},
    {'step': 'chronological_split',
     'detail': f"val = rows with Date >= max(Date) - {VAL_WEEKS} weeks; identical to the XGBoost/LightGBM notebooks"},
    {'step': 'no_series_filter_on_scoring',
     'detail': 'every Store-Dept pair in the validation window is scored; short pairs get a median fallback'},
    {'step': 'context_reindex',
     'detail': "asfreq('W-FRI') per series over the fit portion, ending at FIT_END; gaps interpolated"},
    {'step': 'interpolation_is_input_only',
     'detail': 'interpolated weeks are model input only; every scored row is a real observation'},
    {'step': 'windowing',
     'detail': f'sliding ({INPUT_LEN} -> {OUTPUT_LEN}) windows built from the fit portion only'},
    {'step': 'monitor_holdout',
     'detail': 'last fit window of each series reserved as the early-stopping monitor set'},
    {'step': 'instance_normalization', 'detail': 'RevIN inside the model; no global scaler is fitted'},
    {'step': 'fallback_aggregates',
     'detail': 'fit-portion median (store-dept -> dept -> global) for pairs with no context window'},
]


# ---------- run 1: cleaning / preprocessing ----------
with mlflow.start_run(run_name='DLinear_Cleaning'):
    mlflow.set_tag('stage', 'cleaning')
    mlflow.log_params({
        'input_len': INPUT_LEN,
        'output_len': OUTPUT_LEN,
        'val_weeks': VAL_WEEKS,
        'resample_freq': 'W-FRI',
        'gap_fill': 'interpolate + bfill + ffill (context only)',
        'scaler': 'RevIN (per-window, inside the model)',
        'n_preprocessing_steps': len(PREPROCESSING_STEPS),
    })
    mlflow.log_metrics({
        'train_rows': train.shape[0],
        'train_cols': train.shape[1],
        'train_missing_cells': int(train.isna().sum().sum()),
        'n_stores': int(train['Store'].nunique()),
        'n_depts': int(train['Dept'].nunique()),
        'n_store_dept_pairs': int(train[['Store','Dept']].drop_duplicates().shape[0]),
        'fit_rows': len(gbm_fit),
        'val_rows': len(gbm_val),
        'val_dates': len(val_dates),
        'val_pairs': int(gbm_val[['Store','Dept']].drop_duplicates().shape[0]),
        'n_series_with_context': len(series_ctx),
        'n_context_interpolated_points': n_ctx_interpolated,
        'train_windows': X_train.shape[0],
        'monitor_windows': X_monitor.shape[0],
    })
    mlflow.log_dict({'steps': PREPROCESSING_STEPS}, 'preprocessing/pipeline.json')
    mlflow.log_dict({'columns': train.columns.tolist()}, 'preprocessing/processed_columns.json')
    mlflow.log_input(
        mlflow.data.from_pandas(train, name='train_processed', targets='Weekly_Sales'),
        context='training',
    )

# ---------- run 2: training (early stopping on the monitor set) ----------
with mlflow.start_run(run_name='DLinear_Training'):
    mlflow.set_tag('stage', 'training')
    mlflow.log_params({
        'architecture': 'DLinear (series decomposition + one linear head per component)',
        'kernel_size': KERNEL_SIZE,
        'normalization': 'RevIN (deviates from the paper, which standardizes globally)',
        'loss': 'HuberLoss(delta=1.0)',
        'optimizer': 'AdamW',
        'learning_rate': 5e-4,
        'weight_decay': 1e-2,
        'scheduler': 'CosineAnnealingLR',
        'batch_size': 1024,
        'grad_clip_norm': 0.5,
        'max_epochs': N_EPOCHS,
        'early_stopping_patience': patience,
        'device': device,
    })

    # The loss curve, replayed epoch by epoch so DagsHub renders it as a chart.
    for h in history:
        mlflow.log_metrics(
            {'train_loss': h['train_loss'], 'val_loss': h['val_loss'], 'lr': h['lr']},
            step=h['epoch'],
        )

    mlflow.log_metrics({
        'best_val_loss': best_val_loss,
        'best_epoch': best_epoch,
        'epochs_trained': epochs_trained,
        'n_parameters': sum(p.numel() for p in model_dlinear.parameters()),
    })

    with tempfile.TemporaryDirectory() as tmp:
        p = os.path.join(tmp, 'training_history.csv')
        pd.DataFrame(history).to_csv(p, index=False)
        mlflow.log_artifact(p, artifact_path='training')

# ---------- run 3: validation (chronological holdout) ----------
with mlflow.start_run(run_name='DLinear_Validation'):
    mlflow.set_tag('stage', 'validation')
    mlflow.set_tag('population', 'full' if population_complete else 'partial')
    mlflow.log_params({
        'scheme': 'chronological holdout (no k-fold: the target is a time series), rebuilt exactly as in the XGBoost/LightGBM notebooks',
        'val_weeks': VAL_WEEKS,
        'val_start': str(gbm_val['Date'].min().date()),
        'val_end': str(gbm_val['Date'].max().date()),
        'forecast_strategy': f'roll the {OUTPUT_LEN}-week head forward {N_BLOCKS}x, feeding predictions back as context, truncated to {len(val_dates)} weeks',
        'fallback': 'fit-portion median (store-dept -> dept -> global) for pairs with no context window',
        'population_complete': population_complete,
        'scored_on': 'same rows as XGBoost/LightGBM: directly comparable',
    })
    mlflow.log_metrics({
        'baseline_wmae': baseline_wmae,
        'final_wmae': dlinear_wmae,
        'val_wmae_holiday': dlinear_wmae_holiday,
        'val_wmae_non_holiday': dlinear_wmae_non_holiday,
        'improvement_over_baseline': baseline_wmae - dlinear_wmae,
        'val_rows': len(scored),
        'n_model_forecasts': n_model,
        'n_median_fallbacks': n_fallback,
        'per_pair_wmae_median': float(per_pair['wmae'].median()),
        # WMAE is absolute-dollar, so it tracks series volume. Logged so the
        # score can never be read as if it were scale-free.
        'per_pair_wmae_vs_volume_corr': float(per_pair['wmae'].corr(per_pair['mean_weekly_sales'])),
    })
    with tempfile.TemporaryDirectory() as tmp:
        p = os.path.join(tmp, 'per_pair_wmae.csv')
        per_pair.to_csv(p, index=False)
        mlflow.log_artifact(p, artifact_path='validation')

# ---------- run 4: final model ----------
with mlflow.start_run(run_name='DLinear_Final_Model') as final_run:
    mlflow.set_tags({'stage': 'final_model', 'algorithm': 'dlinear'})
    mlflow.log_params({
        'input_len': INPUT_LEN,
        'output_len': OUTPUT_LEN,
        'kernel_size': KERNEL_SIZE,
        'n_series_with_context': len(series_ctx),
        'best_epoch': best_epoch,
    })
    mlflow.log_metrics({
        'final_wmae': dlinear_wmae,
        'baseline_wmae': baseline_wmae,
        'best_val_loss': best_val_loss,
    })

    # Unlike the GBM notebooks this model is not registered: it consumes a
    # (batch, 52) window of past sales, not the raw Kaggle test frame, so it is
    # not interchangeable with the WalmartSalesForecast pipeline versions.
    model_info = mlflow.pytorch.log_model(
        model_dlinear,
        artifact_path='model',
        # cloudpickle embeds the notebook-defined nn.Module in the artifact;
        # the default pickler stores it as a __main__ reference instead
        pickle_module=cloudpickle,
        input_example=X_monitor[:5],
    )
    mlflow.log_artifact('models/dlinear_best.pkl', artifact_path='estimator')
    print('DLinear final run:', final_run.info.run_id)